In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: פונקציית Qiskit מאת Qedma
*ראה את [ה-API reference](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan, Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בשלב טרום-הוצאה לשוק וכפופות לשינויים.

## סקירה כללית
בעוד שיחידות עיבוד קוונטיות השתפרו מאוד בשנים האחרונות, שגיאות הנובעות מרעש ופגמים בחומרה הקיימת נותרות אתגר מרכזי עבור מפתחי אלגוריתמים קוונטיים. ככל שהתחום מתקרב לחישובים קוונטיים בקנה מידה של תועלת שלא ניתן לאמת קלאסית, פתרונות לביטול רעש בדיוק מובטח הופכים חשובים יותר ויותר. כדי להתגבר על אתגר זה, Qedma פיתחה את Quantum Error Mitigation (QESEM), המשולב בצורה חלקה ב-IBM Quantum Platform כ-[פונקציית Qiskit](/guides/functions).

עם QESEM, משתמשים יכולים להריץ את ה-Circuits הקוונטיים שלהם על QPUs רועשים ולקבל תוצאות מדויקות ללא שגיאות עם תקורות זמן QPU יעילות ביותר, קרוב לגבולות היסודיים. כדי להשיג זאת, QESEM מנצל חבילה של שיטות קנייניות שפותחו על ידי Qedma לאפיון והפחתת שגיאות. טכניקות הפחתת שגיאות כוללות אופטימיזציית Gate, Transpilation מודע לרעש, דיכוי שגיאות (ES) ומיתון שגיאות לא מוטה (EM). עם שילוב שיטות מבוססות-אפיון אלו, משתמשים יכולים להשיג תוצאות אמינות ללא שגיאות עבור Circuits קוונטיים גדולים וגנריים, ולפתוח יישומים שלא ניתן להשיגם אחרת.

לתיאור מלא של הרכיבים הבסיסיים, כמו גם הדגמה בקנה מידה של תועלת, ראה במאמר [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## תיאור
תוכל להשתמש בפונקציית QESEM של Qedma כדי להעריך ולבצע בקלות את ה-Circuits שלך עם דיכוי ומיתון שגיאות, תוך השגת נפחי Circuit גדולים יותר ודיוק גבוה יותר. לשימוש ב-QESEM, עליך לספק Circuit קוונטי, קבוצה של observables למדידה, דיוק סטטיסטי יעד לכל observable, ו-QPU שנבחר. לפני הרצת ה-Circuit לדיוק היעד, תוכל להעריך את זמן ה-QPU הנדרש בהתבסס על חישוב אנליטי שאינו מחייב ביצוע Circuit. לאחר שתהיה מרוצה מהערכת זמן ה-QPU, תוכל להריץ את ה-Circuit עם QESEM.

כאשר אתה מריץ Circuit, QESEM מבצע פרוטוקול אפיון מכשיר המותאם ל-Circuit שלך, ומניב מודל רעש אמין עבור השגיאות המתרחשות ב-Circuit. בהתבסס על האפיון, QESEM מיישם תחילה Transpilation מודע לרעש כדי למפות את ה-Circuit הקלט על קבוצת qubits ו-Gates פיזיים, אשר ממזערת את הרעש המשפיע על ה-observable היעד. אלה כוללים את ה-Gates הזמינים ילידים (CX/CZ על מכשירי IBM&reg;), כמו גם Gates נוספות שאופטמו על ידי QESEM, ויוצרים את מערך ה-Gate המורחב של QESEM. לאחר מכן QESEM מריץ קבוצה של Circuits ES ו-EM מבוססי-אפיון על ה-QPU ואוסף את תוצאות המדידה שלהם. אלה עוברים לאחר מכן עיבוד קלאסי כדי לספק ערך ציפייה לא מוטה ופס שגיאה לכל observable, המתאים לדיוק המבוקש.

![סקירת Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM הוכח כמספק תוצאות בדיוק גבוה עבור מגוון יישומים קוונטיים ובנפחי Circuit הגדולים ביותר הניתנים להשגה כיום. QESEM מציע את התכונות הבאות הפונות למשתמש, המוצגות בסעיף הבנצ'מרקים להלן:
-	**דיוק מובטח:** QESEM מפיק הערכות לא מוטות לערכי ציפייה של observables. שיטת ה-EM שלו מצוידת בערבויות תיאורטיות, אשר - יחד עם האפיון החדשני של Qedma - מבטיחות שהמיתון מתכנס לפלט Circuit ללא רעש עד לדיוק המוגדר על ידי המשתמש. בניגוד לשיטות EM היוריסטיות רבות הנוטות לשגיאות שיטתיות או הטיות, הדיוק המובטח של QESEM חיוני להבטחת תוצאות אמינות ב-Circuits ו-observables גנריים.
-	**מדרגיות ל-QPUs גדולים:** זמן ה-QPU של QESEM תלוי בנפחי Circuit, אך אינו תלוי במספר ה-Qubits. Qedma הדגים את QESEM על המכשירים הקוונטיים הגדולים ביותר הזמינים כיום, כולל IBM Quantum Eagle בן 127 qubit ומכשירי Heron בני 133 qubit.
-	**אגנוסטי ליישום:** QESEM הוכח על מגוון יישומים, כולל סימולציית Hamiltonian, VQE, QAOA ואמידת משרעת. משתמשים יכולים להזין כל Circuit קוונטי ו-observable למדידה, ולקבל תוצאות מדויקות ללא שגיאות. המגבלות היחידות נקבעות על ידי מפרטי החומרה וזמן ה-QPU המוקצה, הקובעים את נפחי ה-Circuit הנגישים ואת דיוק הפלט. בניגוד לכך, פתרונות הפחתת שגיאות רבים ספציפיים ליישום או כוללים היוריסטיקה בלתי נשלטת, מה שהופך אותם לבלתי ישימים ל-Circuits ויישומים קוונטיים גנריים.
-  **מערך Gate מורחב:** QESEM תומך ב-Gates עם זוויות שבריות, ומספק Gates $Rzz(\theta)$ אופטמיזציה על ידי Qedma על מכשירי IBM Quantum Heron ו-Eagle. מערך Gate מורחב זה מאפשר קומפילציה יעילה יותר ופותח נפחי Circuit גדולים יותר פי 2 בהשוואה לקומפילציית CX/CZ ברירת מחדל.
-	**Multibase observables:** QESEM תומך ב-observables קלט המורכבים ממחרוזות Pauli רבות שאינן מתחלפות, כגון Hamiltonians גנריים. בחירת בסיסי המדידה ואופטימיזציית הקצאת משאבי ה-QPU (shots ו-Circuits) מבוצעת אז באופן אוטומטי על ידי QESEM כדי למזער את זמן ה-QPU הנדרש לדיוק המבוקש. אופטימיזציה זו, המתחשבת בנאמנויות חומרה וקצבי ביצוע, מאפשרת לך להריץ Circuits עמוקים יותר ולהשיג דיוק גבוה יותר.
## בנצ'מרקים
QESEM נבדק על מגוון רחב של מקרי שימוש ויישומים. הדוגמאות הבאות יכולות לסייע לך להעריך אילו סוגי עומסי עבודה ניתן להריץ עם QESEM.

מדד ביצועים מרכזי לכימות קושי מיתון השגיאות וסימולציה קלאסית עבור Circuit ו-observable נתונים הוא **נפח פעיל**: מספר ה-Gate CNOT המשפיעים על ה-observable ב-Circuit. הנפח הפעיל תלוי בעומק ורוחב ה-Circuit, במשקל ה-observable ובמבנה ה-Circuit, הקובע את חרוט האור של ה-observable. לפרטים נוספים, ראה את ההרצאה מ-[IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM מספק ערך גדול במיוחד במשטר הנפח הגבוה, ומספק תוצאות אמינות ל-Circuits ו-observables גנריים.

![נפח פעיל](../docs/images/guides/qedma-qesem/active_volume.svg)

| יישום    | מספר qubits | מכשיר | תיאור Circuit | דיוק | זמן כולל | שימוש Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE circuit  | 8              | Eagle (r3) | 21 שכבות סה"כ, 9 בסיסי מדידה, שרשרת 1D                    | 98%      | 35 דקות      | 14 דקות         |
| Kicked Ising   | 28              | Eagle (r3) | 3 שכבות ייחודיות x 3 צעדים, טופולוגיית heavy-hex דו-ממדית                      | 97%     | 22 דקות      | 4 דקות          |
| Kicked Ising   | 28              | Eagle (r3) | 3 שכבות ייחודיות x 8 צעדים, טופולוגיית heavy-hex דו-ממדית                     | 97%      | 116 דקות      | 23 דקות          |
| סימולציית Hamiltonian Trotterized   | 40  | Eagle (r3)            | 2 שכבות ייחודיות x 10 צעדי Trotter, שרשרת 1D                    | 97%      | 3 שעות     | 25 דקות         |
| סימולציית Hamiltonian Trotterized   | 119  | Eagle (r3)           | 3 שכבות ייחודיות x 9 צעדי Trotter, טופולוגיית heavy-hex דו-ממדית                    | 95%      | 6.5 שעות     | 45 דקות         |
| Kicked Ising  | 136             | Heron (r2) | 3 שכבות ייחודיות x 15 צעדים, טופולוגיית heavy-hex דו-ממדית                    | 99%      | 52 דקות             | 9 דקות           |

הדיוק נמדד כאן ביחס לערך האידיאלי של ה-observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, כאשר '$\epsilon$' הוא הדיוק האבסולוטי של המיתון (שנקבע על ידי קלט המשתמש), ו-$\langle O \rangle_{ideal}$ הוא ה-observable ב-Circuit ללא רעש.
'שימוש Runtime' מודד את השימוש של הבנצ'מרק במצב batch (סכום על שימוש של עבודות בודדות), ואילו 'זמן כולל' מודד שימוש במצב session (זמן קיר הניסוי), הכולל זמני חישוב ותקשורת קלאסיים נוספים. QESEM זמין לביצוע בשני המצבים, כך שמשתמשים יכולים לעשות את השימוש הטוב ביותר במשאביהם הזמינים.

ה-Circuits של Kicked Ising בן 28 qubit מדמים את Discrete Time Quasicrystal שנחקר על ידי Shinjo et al. (ראה [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) ו-[Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) על שלושה לולאות מחוברות של ibm_kawasaki. פרמטרי ה-Circuit שנלקחו כאן הם $(\theta_x, \theta_z) = (0.9 \pi, 0)$, עם מצב ראשוני פרומגנטי $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. ה-observable הנמדד הוא הערך המוחלט של המגנטיזציה $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. ניסוי Kicked Ising בקנה מידה של תועלת הופעל על 136 ה-Qubits הטובים ביותר של ibm_fez; בנצ'מרק ספציפי זה הופעל בזווית Clifford $(\theta_x, \theta_z) = (\pi, 0)$, שבה הנפח הפעיל גדל לאט עם עומק ה-Circuit, מה שמאפשר - יחד עם נאמנויות מכשיר גבוהות - דיוק גבוה בזמן ריצה קצר.

ה-Circuits של סימולציית Hamiltonian Trotterized הם עבור מודל Transverse-Field Ising בזוויות שבריות: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ ו-$(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ בהתאמה (ראה [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). ה-Circuit בקנה מידה של תועלת הופעל על 119 ה-Qubits הטובים ביותר של ibm_brisbane, ואילו ניסוי ה-40 qubit הופעל על השרשרת הטובה הזמינה. הדיוק מדווח עבור המגנטיזציה; תוצאות בדיוק גבוה הושגו גם עבור observables במשקל גבוה יותר.

ה-Circuit של VQE פותח יחד עם חוקרים ממרכז לטכנולוגיה וישומים קוונטיים ב-Deutsches Elektronen-Synchrotron (DESY). ה-observable היעד כאן היה Hamiltonian המורכב ממספר רב של מחרוזות Pauli שאינן מתחלפות, המדגיש את הביצועים האופטימיזציים של QESEM עבור observables מרובי בסיסים. המיתון הוחל על ansatz שעבר אופטימיזציה קלאסית; למרות שתוצאות אלה עדיין לא פורסמו, תוצאות באותה איכות יתקבלו עבור Circuits שונים עם מאפיינים מבניים דומים.
## תחילת עבודה
אמת את זהותך באמצעות [מפתח ה-API של IBM Quantum Platform](http://quantum.cloud.ibm.com/), ובחר את פונקציית Qiskit של QESEM כדלקמן. (קטע קוד זה מניח שכבר [שמרת את החשבון שלך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

תוכל להשתמש בממשקי ה-API המוכרים של Qiskit Serverless כדי לבדוק את סטטוס עומס העבודה של פונקציית Qiskit שלך או להחזיר תוצאות:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

קטע הקוד הבא מתאר כיצד לאחזר את הערכת זמן ה-QPU (כאשר `estimate_time_only` מוגדר):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


קטע הקוד הבא מדגים כיצד לאחזר את תוצאות המיתון (כאשר `estimate_time_only` אינו מוגדר) ומדדי ביצוע. אלה מכילים נתונים חיוניים המאפשרים הבנה עמוקה יותר של כיצד פרמטרים שונים משפיעים על ביצוע QESEM. זה עשוי להיות רלוונטי גם בעת כתיבת מאמר המבוסס על המחקר שלך.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## אחזור הודעות שגיאה
אם סטטוס עומס העבודה שלך הוא ERROR, השתמש ב-`job.result()` כדי לאחזר את הודעת השגיאה כדלקמן:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## קבלת תמיכה

צוות התמיכה של Qedma כאן לעזור! אם נתקלת בבעיות כלשהן או יש לך שאלות לגבי שימוש בפונקציית QESEM של Qiskit, אל תהסס לפנות. צוות התמיכה הידידותי והמיומן שלנו מוכן לסייע לך בכל דאגה טכנית או פנייה שיש לך.

תוכל לשלוח אלינו אימייל לכתובת support@qedma.com לקבלת סיוע. כלול כמה שיותר פרטים על הבעיה שאתה חווה כדי לעזור לנו לספק תגובה מהירה ומדויקת. תוכל גם ליצור קשר עם נציג ה-POC הייעודי שלך ב-Qedma באמצעות אימייל או טלפון.

כדי לסייע לנו לטפל בך ביעילות רבה יותר, ספק את המידע הבא כאשר אתה יוצר קשר:

- תיאור מפורט של הבעיה
- מזהה העבודה
- הודעות שגיאה או קודים רלוונטיים

אנחנו מחויבים לספק לך תמיכה מהירה ויעילה כדי להבטיח שתהיה לך החוויה הטובה ביותר עם פונקציית Qiskit שלנו.

אנחנו תמיד שואפים לשפר את המוצר שלנו ומברכים על ההצעות שלך! אם יש לך רעיונות כיצד נוכל לשפר את השירותים או תכונות שתרצה לראות, שלח לנו את מחשבותיך לכתובת support@qedma.com.

## הצעדים הבאים

> **Tip:** - [בקש גישה ל-Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - בקר ב-[API reference](https://docs.quantum.ibm.com/api/functions/qedma-qesem) לפונקציית Qiskit זו.
> - ראה ב-[Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - ראה ב-[Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).